In [1]:
from konlpy.tag import Mecab
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from tensorflow.keras.preprocessing.text import Tokenizer, text_to_word_sequence
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import Dense, Flatten, Embedding
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.preprocessing.sequence import pad_sequences

2025-05-23 09:18:17.020078: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-05-23 09:18:17.311935: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1747959497.418957     505 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1747959497.448875     505 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1747959497.699581     505 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking 

In [2]:
test_data=pd.read_csv("../05machine_learning/data/bank_app_reviews_test.csv")
test_data.head()

,리뷰일,평점,사용자리뷰,업체답변,은행명
0,2024-02-08,5,고경민계장님감사해요,"안녕하세요 최순녀 고객님. 칭찬 진심으로 감사드리며, 더욱 편리하고 안정적인 서비스...",우리
1,2023-07-24,5,저축목표피드 새로 생긴거 너무좋은데 분명 카테고리를 저축으로 했는데 왜 인식이 안되...,"신아​ 님, 안녕하세요? 뱅크샐러드 고객감동팀​입니다. 소중한 시간내어 고객센터에 ...",뱅크샐러드
2,2023-09-25,1,아니 이딴걸 편리하게 사용하는앱이라고 쳐만들엇나 이렇게 불편하게만든건 일부러그런거에...,안녕하세요. 우리은행입니다. 먼저 우리WON뱅킹 이용에 불편을 드려 죄송합니다. 보...,우리
3,2024-02-15,3,몇 년째 만족하며 사용중이라 조금식 개선되어거는 모습에 만족하며 사용중입니다. 하지...,안녕하세요? 뱅크샐러드 고객감동팀입니다. 뱅크샐러드에 KB pay를 연결해 모든 자...,뱅크샐러드
4,2023-06-19,5,스타뱅킹을 사용 하고나서부터 편안해서 좋아요,"한송림 고객님, 안녕하세요! KB스타뱅킹을 이용해 주셔서 진심으로 감사드립니다. 앞...",국민


In [3]:
# 특수문자 제거
import re

def clean_text(text):
    cleaned=re.sub(r'[^가-힣a-zA-Z0-9\s]','', text)
    cleaned=re.sub(r'\s+','',cleaned)
    return cleaned.strip()

In [4]:
test_data['사용자리뷰']=test_data['사용자리뷰'].apply(clean_text)
test_data['사용자리뷰']

0                                              고경민계장님감사해요
1       저축목표피드새로생긴거너무좋은데분명카테고리를저축으로했는데왜인식이안되는걸까요저축액인식되...
2                아니이딴걸편리하게사용하는앱이라고쳐만들엇나이렇게불편하게만든건일부러그런거에요
3       몇년째만족하며사용중이라조금식개선되어거는모습에만족하며사용중입니다하지만요즘kb앱의kbp...
4                                    스타뱅킹을사용하고나서부터편안해서좋아요
                              ...                        
9529    만보기이벤트는실망스러워요후기말투다똑같고사기맞죠양심이참정직하게확률을알려주세요차라리10...
9530                           기능이많아다사용해보진못했지만대체적으로편한거같아요
9531                                                편리하네요
9532                                             사용하기편리해요
9533    너무후져서오랜만에어플이용하다가욕했답니다인증방식이2010년대에머물러계심여기5점짜리는지...
Name: 사용자리뷰, Length: 9534, dtype: object

In [5]:
test_data['is_good']=test_data['평점'].apply(lambda x: 1 if x>=4 else 0)
test_data['is_good']

0       1
1       1
2       0
3       0
4       1
       ..
9529    0
9530    1
9531    1
9532    1
9533    0
Name: is_good, Length: 9534, dtype: int64

In [6]:
mecab=Mecab()

In [7]:
tokenized_docs=test_data['사용자리뷰'].apply(mecab.morphs)

# train에서 사용했던 tokenizer를 불러와서 one hot encoding

In [8]:
import joblib

In [9]:
token=joblib.load("./model/bank_app_tokenizer.joblib")

In [10]:
x=token.texts_to_sequences(tokenized_docs)
print(x[0])

[6248, 327, 111, 71]


# train에서 사용했던 패딩 길이(모델에 넣을 컬럼 수)

In [11]:
max_length=joblib.load("./model/bank_app_max_length.joblib")

In [12]:
X_padded=pad_sequences(x, maxlen=max_length, padding='post')

홀드아웃

In [13]:
y = test_data['is_good']
y

0       1
1       1
2       0
3       0
4       1
       ..
9529    0
9530    1
9531    1
9532    1
9533    0
Name: is_good, Length: 9534, dtype: int64

In [14]:
birnn_best = load_model("./model/bank_app_review_birnn.keras")
cnn_lstm_best = load_model("./model/bank_app_review_lstm_cnn.keras")
attn_best = load_model("./model/bank_app_review_attn_model.keras")

I0000 00:00:1747959505.735989     505 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 1347 MB memory:  -> device: 0, name: NVIDIA GeForce MX450, pci bus id: 0000:01:00.0, compute capability: 7.5


ValueError: File not found: filepath=./model/bank_app_review_lstm_cnn.keras. Please ensure the file is an accessible `.keras` zip file.

In [ ]:
birnn_pred = birnn_best.predict(X_padded)
cnn_latm_pred = cnn_lstm_best.predict(X_padded)
attn_pred = attn_best.predict(X_padded)

In [ ]:
birnn_pred = pd.DataFrame(birnn_pred)
cnn_lstm_pred = pd.DataFrame(cnn_latm_pred)
attn_pred = pd.DataFrame(attn_pred)

In [ ]:
birnn_result = y.join(birnn_pred)
cnn_lstm_result = y.join(cnn_lstm_pred)
attn_pred_result = y.join(attn_pred)

In [ ]:
birnn_result.loc[:, 0] = birnn_result.loc[:, 0].apply(lambda x: 1 if x > 0.5 else 0)
cnn_lstm_result.loc[:, 0] = cnn_lstm_result.loc[:, 0].apply(lambda x: 1 if x > 0.5 else 0)
attn_pred_result.loc[:, 0] = attn_pred_result.loc[:, 0].apply(lambda x: 1 if x > 0.5 else 0)

In [ ]:
from sklearn.metrics import classification_report

In [ ]:
print(classification_report(birnn_result['is_good'], birnn_result[0]))

In [ ]:
print(classification_report(cnn_lstm_result['is_good'], cnn_lstm_result[0]))

In [ ]:
print(classification_report(attn_pred_result['is_good'], attn_pred_result[0]))

# evaluate

In [ ]:
%%time
birnn_best.evaluate(X_padded, test_data['is_good'])

In [ ]:
%%time
cnn_lstm_best.evaluate(X_padded, test_data['is_good'])

In [ ]:
%%time
attn_best.evaluate(X_padded, test_data['is_good'])